In [2]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["DOCLING_DEVICE"] = "cpu"

from transformers import AutoTokenizer
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.chunking import HybridChunker
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

FILE_PATH = "./test1.pdf"
TOKENIZER_ID = "nomic-ai/nomic-embed-text-v2-moe" 
OLLAMA_EMBED_ID = "nomic-embed-text-v2-moe" 

def build_ingestion_pipeline():
    # 1. Setup the Tokenizer for accurate chunk limits
    print("1. Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, local_files_only=False)

    # 2. Configure Docling Pipeline (Merging your OCR snippet!)
    print("2. Configuring Docling OCR and Layout Parser...")
    pipeline_options = PdfPipelineOptions()
    
    # Enable OCR for scanned PDFs (Falls back to EasyOCR/Tesseract)
    pipeline_options.do_ocr = True 
    
    # Enable image extraction for your multi-modal Vision LLM later
    pipeline_options.generate_picture_images = True 
    
    # Force CPU processing to prevent Mac GPU crashing on heavy OCR tasks
    pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=4, 
        device=AcceleratorDevice.CPU 
    )

    # Register the pipeline options
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    # 3. Parse PDF
    print(f"3. Parsing PDF (this may take longer due to OCR): {FILE_PATH}...")
    parsed_doc = converter.convert(FILE_PATH).document

    # 4. Hybrid Chunking
    print("4. Executing Token-Aware Hybrid Chunking...")
    chunker = HybridChunker(
        tokenizer=tokenizer,
        max_tokens=512,
        merge_peers=True
    )
    docling_chunks = list(chunker.chunk(parsed_doc))

    # 5. Bridge to LlamaIndex
    print("5. Converting to LlamaIndex Nodes...")
    llama_nodes = []
    
    for i, chunk in enumerate(docling_chunks):
        node = TextNode(
            text=chunk.text,
            metadata={
                "source_file": FILE_PATH,
                "chunk_index": i,
                "has_ocr": True # Just a flag to remember this pipeline used OCR
            }
        )
        llama_nodes.append(node)

    print(f"✅ Success! Created {len(llama_nodes)} LlamaIndex Nodes.")
    return llama_nodes


# --- LlamaIndex Execution ---

# Point LlamaIndex to your local Ollama models
Settings.llm = Ollama(model="gemma4:e4b ", request_timeout=300.0)
Settings.embed_model = OllamaEmbedding(model_name=OLLAMA_EMBED_ID)

# Run the pipeline
nodes = build_ingestion_pipeline()

if nodes:
    print("6. Building Vector Store Index...")
    index = VectorStoreIndex(nodes)

    print("\n--- Testing Retrieval ---")
    query_engine = index.as_query_engine()
    response = query_engine.query("What are the questions asked in this paper?")
    print(response)

1. Loading Tokenizer...
2. Configuring Docling OCR and Layout Parser...
3. Parsing PDF (this may take longer due to OCR): ./test1.pdf...


[INFO] 2026-05-29 17:20:02,773 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-29 17:20:02,777 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-29 17:20:02,788 [RapidOCR] download_file.py:60: File exists and is valid: /Users/raziqs/Desktop/PaperSloth/PaperSloth/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-29 17:20:02,789 [RapidOCR] main.py:50: Using /Users/raziqs/Desktop/PaperSloth/PaperSloth/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-29 17:20:02,937 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-29 17:20:02,938 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-29 17:20:02,940 [RapidOCR] download_file.py:60: File exists and is valid: /Users/raziqs/Desktop/PaperSloth/PaperSloth/.venv/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-29 17:20:02,940 [RapidOCR] main.py:50: Using /Users/ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7660 > 512). Running this sequence through the model will result in indexing errors


4. Executing Token-Aware Hybrid Chunking...
5. Converting to LlamaIndex Nodes...
✅ Success! Created 73 LlamaIndex Nodes.
6. Building Vector Store Index...

--- Testing Retrieval ---
The questions included in the paper are:

1.  **Explain the following terminologies:**
    *   Measuring absolute pressure (psia).
    *   Measuring gauge pressure (psig).
    *   Measuring differential pressure (psi).
2.  **Assess the following rejuvenation:**
    *   Requirement to convert a wired system to a 5G system.
    *   The benefit of using 5G transmission.
3.  **Analyze:** Determine whether a PI controller can achieve zero offset due to a unit step change at the Setpoint, ASP(s), as $t \to \infty$, given a general model of a closed-loop system shown in FIGURE Q4, where the general transfer function is provided.
